<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/P21_51_Chart2LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ॐ श्रीश्री महाकाली नमः

![alt text](http://1.bp.blogspot.com/_5QlanosqxiQ/SXLOLTu7S7I/AAAAAAAAAm8/0r6C7lCxsic/S1600-R/p21-header-003.png) <br>


---



[Prithwis Mukerjee](http://www.yantrajaal.com) ||
Website - [Parashar21](https://parashar21.blogspot.com)<br>

Copyright (c) 2022, Prithwis Mukerjee All rights reserved.

This source code is licensed under the GNU GPL v3.0 -style license found in the LICENSE file in the root directory of this source tree.

#Version 51 | Parashar21 > LLMs

##Rationale

![Parashar21 Architecture](https://raw.githubusercontent.com/prithwis/parashar21/main/images/p21-Arch-2026.png)<br>

The core engine of **Parashar21** is designed to generate a complete Jyotisha Chart (Rashi/Natal or Navamsha) from the native's **Date of Birth (DoB), Time of Birth (ToB)** and **Place of Birth (PoB)** using the Python implementation of the Swiss Ephemeris.

The computed horoscope is first organised into structured **CSV** files and then transformed into a comprehensive **JSON** representation containing planetary positions, house lords, aspects, conjunctions, bhavas, dignities, Ashtakavarga points, Vimshottari Dasha, Gochar analysis and other derived astrological parameters. This JSON data is subsequently loaded into **Pandas** DataFrames, which form the common working model for all subsequent analysis.

From this unified data model, Parashar21 generates two complementary outputs:

* A detailed **Microsoft Word** report intended for human readers.
* A structured **LLM Input** text file designed for Large Language Models.

The LLM Input file contains the complete astrological description of the horoscope in a form suitable for AI-assisted interpretation. Together with selected classical Jyotisha reference texts, it serves as the input to systems such as **Google Gemini NotebookLM**, enabling the generation of transparent, source-based astrological analyses while keeping the computational and interpretive stages clearly separated.
 <br> <br>
For full information about this project, please see the [Parashar21](https://prithwis.github.io/parashar21/) github page.


###Backward Compatibility

This notebook implements the core Parashar21 processing pipeline. Birth data is read from a CSV file, converted into a fully enriched horoscope using the Swiss Ephemeris library, represented internally as a single-row Pandas DataFrame (chart), and then used to generate both human-readable reports and structured inputs for Large Language Models (LLMs). The pipeline has evolved over more than a decade and retains certain design elements from earlier MongoDB-based implementations for reasons of stability and backward compatibility. A detailed description of the overall design, processing stages and architectural decisions is available in ARCHITECTURE.md.

In [20]:
from datetime import datetime
import pytz
print("\033[1m"+'ॐ श्रीश्री महाकाली नमः'+"\033[0m")
print(datetime.now(pytz.timezone('Asia/Calcutta')))

ॐ श्रीश्री महाकाली नमः
2026-08-28 11:15:43.904059+05:30


#SetUp Environment

##Install External Pre Requisites

In [21]:
!python --version
!lsb_release -a
!pip -qq install pyswisseph                                 # https://stackoverflow.com/questions/64277506/pip-install-options-unclear
!pip -qq install python-docx                                # https://python-docx.readthedocs.io/en/latest/


Python 3.13.15
No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy


In [22]:
#Utility functions
#
import pandas as pd
import dateutil
import json
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime
from datetime import timedelta
import pytz
from google.colab import files


## Install, import local components

In [23]:
#Load p21 modules from github
#
!wget -O p21.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/utils/p21.py                  # all global variables
!wget -O p21utils.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/utils/p21utils.py        # large number of utility functions
!wget -O p21utilsR.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/utils/p21utilsR.py      # report writing utility functions
!wget -O p21swe.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/utils/p21swe.py            # Swiss Ephemeris functions
!wget -O p21YogInfo.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/utils/p21YogInfo.py    # yog data - description and conditions
!wget -O p21LLM.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/utils/p21LLM.py            # LLM bridge function

# ---------------------------

#!wget -O Saraswati.png -q https://github.com/prithwis/parashar21/raw/main/images/Saraswati02.png
#!wget -O p21logo.png -q https://github.com/prithwis/parashar21/raw/main/images/p21logo-002-1.png


In [24]:
import p21
import p21utils
import p21utilsR
import p21swe
#import p21YogInfo
import p21LLM
#
# ------------------------------------------------
# required for multiple imports of the same module
# ------------------------------------------------
import importlib
#importlib.reload(p21)
#importlib.reload(p21utils)
#importlib.reload(p21utilsR)
#importlib.reload(p21swe)
#importlib.reload(p21YogInfo)
importlib.reload(p21LLM)
#importlib.reload(p21LLM02)

<module 'p21LLM' from '/content/p21LLM.py'>

In [25]:
# Guess what is happening here :-)
#
!wc *.py | grep total |awk '{print $1 " lines";}'
!sed '/^#/d' *.py | wc |awk '{print $1 " non-comment lines";}'

3124 lines
2819 non-comment lines


## Configure Swiss Ephemeris

In [26]:
#Download Swiss Ephemeris Files
#!wget  http://www.astro.com/ftp/swisseph/ephe/seas_18.se1
!wget -q https://github.com/aloistr/swisseph/raw/master/ephe/seas_18.se1
#!wget  http://www.astro.com/ftp/swisseph/ephe/semo_18.se1
!wget -q https://github.com/aloistr/swisseph/raw/master/ephe/semo_18.se1
#!wget  http://www.astro.com/ftp/swisseph/ephe/sepl_18.se1
!wget  -q https://github.com/aloistr/swisseph/raw/master/ephe/sepl_18.se1
!mkdir ephe
!mv *.se1 ephe
#------------------------------
# Configure SWE with appropriate parameters
#
p21swe.C01_configSWE()

mkdir: cannot create directory ‘ephe’: File exists


#Local Functions

Most of the code is stored in the Local Modules, that are called from these functions stored in this notebook

## Report Generation Function

In [27]:
def genReports(ChartStyle):

    chartData = chart.iloc[0]
    RepID = chart.iloc[0]['pid']['name']+'_'+p21.AnalysisType[0:3]+'_'+ChartStyle[0:1]
    now = datetime.now(pytz.timezone('Asia/Kolkata'))
    fileName = 'p21_'+RepID+'_'+now.strftime("%H%M%S")+'.doc'

    p21utilsR.R511_parseChartData(chartData)                # Break up database data into useful pieces
                                                            # Creates two primary dicts, GLon, GRet
    #p21utilsR.R30_LocateYogs()

    # This is a rather clever hack. During the chart generation, the order in which charts are generated leaves
    # the graha longitudes in the order of the last chart generated
    # Hence there Ashtakvarga points in the output report will carry the points calculated on the basis of the last chart generate
    # and not on the type of analysis that is actually required
    #
    if p21.AnalysisType == 'Rashi' :
        listOfChartTypes = ['Navamsa','Rashi']
    elif p21.AnalysisType == 'Navamsa':
        listOfChartTypes = ['Rashi','Navamsa']
    else:
        print('Error, Unknown Analysis Type')


    for ctype in listOfChartTypes:
        p21.ChartType = ctype
        p21utils.R11_LocateGrahaInRashi()   # Converts Graha Long to Rashi positions
                                            # Rashi Num as well as Rashi name
        if ChartStyle == 'Bengal':
            p21utilsR.R12B_drawChart_Bengal()    # Draw Chart in Bengal style
        if ChartStyle == 'South':
            p21utilsR.R12B_drawChart_South()    # Draw Chart in South style
        if ChartStyle == 'North':
            p21utilsR.R12B_drawChart_North()    # Draw Chart in North style


    # ---------------------------------------------------------
    # Human friendly MS Word document report

    p21utilsR.R01_CreateReportDoc(None,None,'SingleChart')        # Creates MS Word Doc called p21.document for Single or MULTIPLE charts
    #p21utilsR.R01A_CreateReportDoc(RepID)                        # Creates MS Word Doc called p21.document for SINGLE chart, deprecated


    p21utilsR.R512_FormatPage('SingleChart')                    # this is where the basic details of the chart are printed out
    p21utilsR.R512_FormatPage2A()                                # this is where the Vimsottari Dasha details for SINGLE chart scenario
    p21utilsR.R512_FormatPage2B()                                # this is where the AshtakVarga details for SINGLE chart scenario


    p21.document.save(fileName)
    # ------------------------------------

    # LLM friendly txt file

    p21LLM.R601_GenerateLLMInput05()                        # -- the horoscope chart
    if p21.pName != p21.gName:                              # Natal only
        p21LLM.R602_GenerateLLMInput01()                    # -- the vimsottori dasha
    # ------------------------------------
    return fileName

print("Executed at ", datetime.now(pytz.timezone('Asia/Kolkata')))

Executed at  2026-08-28 11:15:53.027711+05:30


## JSON to Pandas
Horoscope charts stored in JSON format are converted into Pandas for subsequent into reports. Historically, the JSON data was stored in MongoDB and then retrieved as Pandas. With the removal of MongoDB, the JSON data is directly converted into Pandas. However the old function has been retained for future clarity

In [28]:
# Deprecated
#def StoreRetrieveChart():
#    with open('peopleData.json') as json_file:
#        oneChart = json.load(json_file)
#
#    db.drop_collection('khona21')
#    db.create_collection('khona21')               # Optional collection creation
#    p21.kollection = db.khona21                   # Set the collection to work with
#
#    # Insert the single chart
#    insert_result = p21.kollection.insert_one(oneChart)
#    return(pd.DataFrame(list(p21.kollection.find({},p21.selCols))))
#
#print("Executed at ", datetime.now(pytz.timezone('Asia/Kolkata')))


In [29]:
def getChartData():
    with open('peopleData.json') as json_file:
        oneChart = json.load(json_file)

    filtered_chart = {}

    # 1. Handle nested 'pid' fields
    if 'pid' in oneChart and isinstance(oneChart['pid'], dict):
        # Extract only the pid sub-fields requested in selCols
        pid_keys = [k.split('.')[1] for k in p21.selCols if k.startswith('pid.')]
        filtered_chart['pid'] = {k: oneChart['pid'][k] for k in pid_keys if k in oneChart['pid']}

    # 2. Extract top-level fields requested in selCols
    top_keys = [k for k, v in p21.selCols.items() if v == 1 and not k.startswith('pid.') and k != '_id']
    for k in top_keys:
        if k in oneChart:
            filtered_chart[k] = oneChart[k]

    # 3. Return as single-row DataFrame
    return pd.DataFrame([filtered_chart])

# Natal Chart Generation



## Sample Natal Data for testing
Data for others can be added by adding an extra line in the cell below. Please follow the same pattern as in the other lines

In [30]:
# Sample Data for some people
#
#%%writefile peopleData.csv
#Gender,DoB_Day,DoB_Mon,DoB_Year,DoB_Time,TZ_OffHours,PoB_Lat,PoB_Lon,TZ_Name,TZ_Type,Name,tag1,tag2,tag3,tag4,tag5,tag6
#M,17,9,1950,23:11,5.5,23.72,63.36,IST,standard,NDModi,Politician,nil,Administrator,nil,nil,nil
#F,10,10,1954,11:00,5.5,13.08,80.27,IST,standard,RekhaG,Actor,nil,Dancer,nil,nil,nil
#M,15,10,1931,1:15,5.5,9.29,79.31,IST,standard,APJAbdulKalam,Engineer,nil,PublicFigure,nil,nil,nil
#F,19,11,1917,23:11,5.5,25.43,81.85,IST,standard,IndiraG,Politician,nil,PublicFigure,nil,nil,nil
#F,19,12,1965,10:26,5.5,22.57,88.37,IST,standard,QueenOH,dummy,nil,dummy,nil,dummy,nil
#M,9,10,1990,23:09,5.5,22.57,88.37,IST,standard,Gentoo,dummy,nil,dummy,nil,dummy,nil
#F,12,9,1990,0:52,5.5,22.87,88.37,IST,standard,TheSaint,dummy,nil,dummy,nil,dummy,nil
#M,4,6,2024,09:05,8.0,1.36,103.82,SST,standard,LionKing,dummy,nil,dummy,nil,dummy,nil
#F,13,2,1992,12:09,5.5,13.08,80.27,IST,standard,ChennaiGirl,dummy,nil,dummy,nil,dummy,nil
#M,4,6,1878,2:00,5.88,26.32,89.45,IST,standard,PrabhatNM,dummy,nil,dummy,nil,dummy,nil -- before 1906, Bengal time was 5:53 mins ahead of GMT
#M,4,4,1919,12:25,5.5,22.87,88.37,IST,standard,SantoshKM,dummy,nil,dummy,nil,dummy,nil
#M,16,7,1898,22:30,5.88,26.32,89.45,IST,standard,SoshiCB,dummy,nil,dummy,nil,dummy,nil -- before 1906, Bengal time was 5:53 mins ahead of GMT
#F,17,6,1928,19:01,5.5,22.57,88.37,IST,standard,RekhaMasi,dummy,nil,dummy,nil,dummy,nil
#M,18,10,1904,19:30,5.88,26.32,89.45,IST,standard,BijoyKB,dummy,nil,dummy,nil,dummy,nil -- before 1906, Bengal time was 5:53 mins ahead of GMT
#M,21,3,1911,1:45,5.5,22.57,88.37,IST,standard,SobhenduM,dummy,nil,dummy,nil,dummy,nil
#M,15,10,1961,23:42,5.5,22.57,88.37,IST,standard,SomCM,dummy,nil,dummy,nil,dummy,nil
#M,16,12,1927,10:00,5.5,22.57,88.37,IST,standard,SubhrenduM,dummy,nil,dummy,nil,dummy,nil -- ToB is between 8:30 and 10
#M,19,6,1933,12:07,5.5,22.57,88.37,IST,standard,NarendraLal,dummy,nil,dummy,nil,dummy,nil
#F,1,5,1938,14:00,5.5,22.87,88.37,IST,standard,RebaM,dummy,nil,dummy,nil,dummy,nil  -- ToB is betwen 12:30 and 3:00
#M,6,11,1975,7:46,5.5,22.87,88.37,IST,standard,Sarbajit,dummy,nil,dummy,nil,dummy,nil

# Known Test Cases -- RoddenAA data from astrobank.wiki

#Gender,DoB_Day,DoB_Mon,DoB_Year,DoB_Time,TZ_OffHours,PoB_Lat,PoB_Lon,TZ_Name,TZ_Type,Name,tag1,tag2,tag3
#M,14,3,1879,11:30,0.66,48.40,10.00,LMT,local mean,C01_AlbertEinstein,nil,nil,nil,nil,nil,nil
#M,24,2,1955,19:15,-8.00,37.78,-122.41,PST,standard,C02_SteveJobs,nil,nil,nil,nil,nil,nil
#M,25,10,1881,23:15,-0.29,36.71,-4.41,LMT,local mean,C03_PabloPicasso,nil,nil,nil,nil,nil,nil
#F,1,6,1926,09:30,-8.00,34.04,-118.25,PST,standard,C04_MarilynMonroe,nil,nil,nil,nil,nil,nil
#M,17,1,1942,18:35,-6.00,38.25,-85.76,CST,standard,C05_MuhammadAli,nil,nil,nil,nil,nil,nil
#F,6,7,1907,08:30,-6.61,19.33,-99.16,LMT,local mean,C06_FridaKahlo,nil,nil,nil,nil,nil,nil
#M,21,7,1899,08:00,-6.00,41.88,-87.78,CST,standard,C07_ErnestHemingway,nil,nil,nil,nil,nil,nil
#M,8,1,1935,04:35,-6.00,34.25,-88.70,CST,standard,C08_ElvisPresley,nil,nil,nil,nil,nil,nil
#M,30,12,1975,22:50,-8.00,33.76,-118.18,PST,standard,C09_TigerWoods,nil,nil,nil,nil,nil,nil
#M,14,12,1946,09:27,5.50,28.66,77.21,IST,standard,C10_SanjayGandhi,nil,nil,nil,nil,nil,nil

#Read Birth Data from a file
#!wget -O peopleData.csv -q https://raw.githubusercontent.com/prithwis/parashar21/main/data/Test5Data.txt # 5 Person Test Data
#!cat peopleData.csv

## Natal Data Preparation
### Only cells after this need to be executed for analysis of additional natives

In [103]:
!rm *.doc
!rm *.json
!rm *.txt



##One line in the next cell needs to be changed

In [104]:
%%writefile peopleData.csv
Gender,DoB_Day,DoB_Mon,DoB_Year,DoB_Time,TZ_OffHours,PoB_Lat,PoB_Lon,TZ_Name,TZ_Type,Name,tag1,tag2,tag3,tag4,tag5,tag6
F,10,10,1954,11:00,5.5,13.08,80.27,IST,standard,RekhaG,Actor,nil,Dancer,nil,nil,nil

Overwriting peopleData.csv


## Natal Data Extraction | Formatting
type of analsis Rashi / Navamsa to be specified here

In [105]:
#%%time

p21.AnalysisType = 'Rashi'                                     # one of ['Rashi','Navamsa']
#p21.AnalysisType = 'Navamsa'                                  # one of ['Rashi','Navamsa']

df = pd.read_csv('peopleData.csv')

p21.ChartType = p21.AnalysisType
# ------------------------------------------------------------
# ------------------------------------------------------------
# Converts date/time info into a detailed horoscope chart
# Stores the same in a JSON file, to be inserted into a local MongoDB database
#
p21swe.C61_Cast2JSON(df)
# ------------------------------------------------------------

#p21utils.SarvaAshtakVarga()                            # Uncomment for debugging Ashtakvarga point calculation

p21.SubMoonLong = p21.GLon['Mo']                        # Preserving Moon Longitude for Gochar
#p21.Subject = p21.pName                                # Preserving Subjects Name for Gochar Chart

p21utils.GetDasha()                                     # Generates Vimsottari Dasha / Antardasha Details


1 records processed, so far
1  records generated and stored in file peopleData.json
p21.SubMoonLong  325.406
Birth Nakshatra  24 P_Bhadrapad


## Natal Chart Generation

In [106]:
%%capture

p21.printDasha = True
chart = getChartData()
ReportFile = genReports('Bengal')

#p21.SubMoonLong = p21.GLon['Mo']                        # Preserving Moon Longitude for Gochar
p21.Subject = p21.pName                                  # Preserving Subjects Name for Gochar Chart

# Note : The MS-Word file that is generated is available in the Colab VM Drive
# and needs to be downloaded to local laptop / machine for viewing in MS-Word


#Gochar Chart

## Gochar Data Preparation

In [107]:
# This cell to be executed ONLY after natal chart has been created, otherwise errors
#
!sed -n 1p peopleData.csv > peopleData2.csv                                  # Copy the Header data
print(p21.gName, p21.SubMoonLong)
now = datetime.now(pytz.timezone('Asia/Kolkata'))
timeTxt = now.strftime("%-d,%-m,%Y,%-H:%-M")

#
# for Gochar chart for any other time, other than now()
#timeTxt = '7,6,2025,9:30'                # any other time in dd,mm,yyyy,hh:mm format will do
#

GocharTxt = 'x,'+timeTxt+',5.5,25.43,81.85,IST,standard,'+p21.gName+',nil,nil,nil,nil,nil,nil'    # Using Lat, Long of Calcutta for Gochar
print(GocharTxt)
with open("peopleData2.csv", 'a') as file1:                                                       # Append Gochar Data
    file1.write(GocharTxt)


_Gochar 325.406
x,28,8,2026,12:15,5.5,25.43,81.85,IST,standard,_Gochar,nil,nil,nil,nil,nil,nil


## Gochar Data Extraction | Formatting

In [108]:
# Load CSV data into Pandas dataframe
#
df = pd.read_csv('peopleData2.csv')

# Should not be changed here ... already defined

#p21.AnalysisType = 'Rashi'                                     # one of ['Rashi','Navamsa']
#p21.AnalysisType = 'Navamsa'                                    # one of ['Rashi','Navamsa']

p21.ChartType = p21.AnalysisType
p21swe.C61_Cast2JSON(df)                                      # For Gochar, the La Long is set to Natal Moon Long
#print(p21.GLon['La'],p21.GLon['Mo'])
#p21utils.SarvaAshtakVarga()                                     # Uncomment for debugging Ashtakvarga point calculation

1 records processed, so far
1  records generated and stored in file peopleData.json


## Gochar Chart Generation

In [109]:
%%capture

p21.printDasha = True
if p21.AnalysisType == 'Rashi':
    chart = getChartData()
    ReportFile = genReports('Bengal')
else :
    print("Gochar report not available with Navamsa analysis\n")
    with open("NoGocharReport.txt", "w") as file:
        file.write("Gochar report not available with Navamsa analysis\n")

# Note : The MS-Word file that is generated is available in the Colab VM Drive
# and needs to be downloaded to local laptop / machine for viewing in MS-Word


In [110]:
print("\033[1m"+'जय पराशर'+"\033[0m")
print(datetime.now(pytz.timezone('Asia/Calcutta')))

जय पराशर
2026-08-28 12:15:46.055137+05:30


#Chronobooks <br>
Three science fiction novels by Prithwis Mukerjee. A dystopian Earth. A technocratic society managed by artificial intelligence. Escape and epiphany on Mars. Can man and machine, carbon and silicon explore and escape into other dimensions of existence? An Indic perspective rooted in Advaita Vedanta and the Divine Feminine.  [More information](http://chronos.yantrajaal.com) <br>
![alt text](https://blogger.googleusercontent.com/img/a/AVvXsEjsZufX_KYaLwAnJP6bUxvDg5RSPn6r8HIZe749nLWX3RuwyshrYEAUpdw03a9WIWRdnzA9epwJOE05eDJ0Ad7kGyfWiUrC2vNuOskb2jA-e8aOZSx8YqzT8mfZi3E4X1Rz3qlEAiv-aTxlCM976BEeTjx4J64ctY3C_FoV4v9aY_U23F8xRqI5Eg=s1600)